# Quantum Teleportation
**Notebook:** Complete 3-qubit protocol — transmits an arbitrary qubit state using a Bell pair and 2 classical bits

## Overview

**Quantum teleportation** transmits an arbitrary qubit state $|\psi\rangle$ from Alice to Bob
using:
- 1 pre-shared **Bell pair** (entangled qubit pair)
- 2 **classical bits** of communication

No quantum channel is needed after the Bell pair is distributed. The state is **not cloned**
(no-cloning theorem is respected) — Alice's qubit is destroyed in the process.

### Protocol (4 steps)
| Step | Action | Description |
|------|--------|-------------|
| 1 | Prepare $\|\psi\rangle$ | Alice encodes the state to teleport on $q_0$ |
| 2 | Bell pair | Entangle $q_1$ (Alice) and $q_2$ (Bob) via $H + \text{CNOT}$ |
| 3 | Bell measurement | Alice measures $q_0, q_1$ jointly → 2 classical bits |
| 4 | Correction | Bob applies $X$ and/or $Z$ based on Alice's bits |

### Why it works
The Bell measurement collapses the joint state into one of 4 Bell states.
Bob's qubit is already in a state related to $|\psi\rangle$ — he just needs to apply
the right Pauli correction (told to him via classical channel) to recover $|\psi\rangle$ exactly.

### Verification
Bob should measure $|0\rangle$ with probability $\cos^2(\theta/2)$ and $|1\rangle$ with $\sin^2(\theta/2)$,
matching the original state $\cos(\theta/2)|0\rangle + \sin(\theta/2)|1\rangle$.

In [ ]:
# Quantum Teleportation Protocol
# Teleports an arbitrary qubit state |ψ⟩ from Alice to Bob using a shared
# Bell pair and 2 classical bits of communication.
#
# Registers:
#   q[0] = message qubit (Alice) — holds the state |ψ⟩ = cos(θ/2)|0⟩ + sin(θ/2)|1⟩
#   q[1] = Alice's half of the Bell pair (entangled with Bob)
#   q[2] = Bob's half of the Bell pair (remote end)
#   c[0], c[1] = classical bits sent from Alice to Bob over a classical channel

theta = pi / 3   # message state parameter — change to teleport a different state

qr = QuantumRegister(3, 'q')
cr = ClassicalRegister(2, 'c')
qc = QuantumCircuit(qr, cr)

# --- Step 1: Prepare message state |ψ⟩ on q[0] ---
# RY(θ)|0⟩ = cos(θ/2)|0⟩ + sin(θ/2)|1⟩  (real-amplitude state on the Bloch sphere)
qc.ry(theta, qr[0])
qc.barrier(label='Prepare |ψ⟩')

# --- Step 2: Create Bell pair |Φ+⟩ between q[1] (Alice) and q[2] (Bob) ---
# H on q[1]: |0⟩ → (|0⟩+|1⟩)/√2  (superposition)
qc.h(qr[1])
# CNOT q[1]→q[2]: entangles the pair → (|00⟩+|11⟩)/√2  (Bell state |Φ+⟩)
qc.cx(qr[1], qr[2])
qc.barrier(label='Bell pair')

# --- Step 3: Bell measurement on Alice's qubits q[0] and q[1] ---
# CNOT q[0]→q[1]: entangles message qubit with Alice's Bell qubit
qc.cx(qr[0], qr[1])
# H on q[0]: rotates into the Bell basis so that measurement gives one of 4 outcomes
qc.h(qr[0])
qc.barrier(label='Bell meas.')
qc.measure(qr[0], cr[0])   # c[0] ← measurement of q[0] (controls Bob's Z correction)
qc.measure(qr[1], cr[1])   # c[1] ← measurement of q[1] (controls Bob's X correction)
qc.barrier(label='Corrections')

# --- Step 4: Bob applies Pauli corrections based on Alice's 2 classical bits ---
# If c[1]=1: Bob applies X (bit-flip) to recover the correct |0⟩/|1⟩ component
with qc.if_test((cr[1], 1)):
    qc.x(qr[2])
# If c[0]=1: Bob applies Z (phase-flip) to recover the correct phase
with qc.if_test((cr[0], 1)):
    qc.z(qr[2])
# After corrections, q[2] is guaranteed to be in state |ψ⟩ = cos(θ/2)|0⟩ + sin(θ/2)|1⟩

qc.draw('mpl')

In [ ]:
# --- Verification circuit ---
# Rebuilds the full protocol and adds a final measurement on Bob's qubit (q[2]).
# We use a separate circuit so we can measure Bob's qubit without disturbing the
# display-only circuit above.
qr2 = QuantumRegister(3, 'q')
cr2 = ClassicalRegister(2, 'c')       # Alice's 2 classical bits
cr_bob = ClassicalRegister(1, 'bob')  # Bob's measurement result
qc_verify = QuantumCircuit(qr2, cr2, cr_bob)

# Step 1: prepare the same message state
qc_verify.ry(theta, qr2[0])

# Step 2: create Bell pair
qc_verify.h(qr2[1])
qc_verify.cx(qr2[1], qr2[2])

# Step 3: Bell measurement
qc_verify.cx(qr2[0], qr2[1])
qc_verify.h(qr2[0])
qc_verify.measure(qr2[0], cr2[0])   # c[0]: controls Z correction
qc_verify.measure(qr2[1], cr2[1])   # c[1]: controls X correction

# Step 4: conditional Pauli corrections on Bob's qubit
with qc_verify.if_test((cr2[1], 1)):
    qc_verify.x(qr2[2])   # X correction if Alice's q[1] measured |1⟩
with qc_verify.if_test((cr2[0], 1)):
    qc_verify.z(qr2[2])   # Z correction if Alice's q[0] measured |1⟩

# Measure Bob's qubit — should reproduce the statistics of |ψ⟩
qc_verify.measure(qr2[2], cr_bob[0])

backend = AerSimulator()
result = backend.run(transpile(qc_verify, backend), shots=2048).result()
counts = result.get_counts()
print("Measurement counts  (format: 'bob c[0]c[1]'):")
print(counts)
plot_histogram(counts, title=f"Teleportation verification (θ={theta:.2f} rad)")

In [ ]:
# --- Theoretical vs measured probabilities ---
# For the state cos(θ/2)|0⟩ + sin(θ/2)|1⟩, the Born rule gives:
#   P(|0⟩) = cos²(θ/2)   P(|1⟩) = sin²(θ/2)
p0_expected = np.cos(theta / 2) ** 2
p1_expected = np.sin(theta / 2) ** 2

# Extract Bob's qubit marginal from the joint count dictionary.
# Qiskit formats the key as "bob_bit alice_c0c1" (space-separated registers,
# leftmost register printed first), so k.split(' ')[0] is Bob's bit.
bob_0 = sum(v for k, v in counts.items() if k.split(' ')[0] == '0')
bob_1 = sum(v for k, v in counts.items() if k.split(' ')[0] == '1')
total = bob_0 + bob_1   # should equal the total number of shots

print(f"Expected:  P(|0⟩) = {p0_expected:.3f},  P(|1⟩) = {p1_expected:.3f}")
print(f"Measured:  P(|0⟩) = {bob_0/total:.3f},  P(|1⟩) = {bob_1/total:.3f}")

# Pass criterion: measured probability within 5% of theoretical (shot-noise tolerance)
success = abs(bob_0 / total - p0_expected) < 0.05
print(f"\nTeleportation {'SUCCESSFUL ✓' if success else 'check failed (try more shots)'}")

In [ ]:
# Theoretical check: Bob should measure |0⟩ with prob cos²(θ/2), |1⟩ with sin²(θ/2)
p0_expected = np.cos(theta / 2) ** 2
p1_expected = np.sin(theta / 2) ** 2

# Extract Bob's qubit marginal from the full count dict
# Key format: "bob alice_c1 alice_c0" (rightmost = cr2, leftmost = cr_bob)
bob_0 = sum(v for k, v in counts.items() if k.split(' ')[0] == '0')
bob_1 = sum(v for k, v in counts.items() if k.split(' ')[0] == '1')
total = bob_0 + bob_1

print(f"Expected:  P(|0⟩) = {p0_expected:.3f},  P(|1⟩) = {p1_expected:.3f}")
print(f"Measured:  P(|0⟩) = {bob_0/total:.3f},  P(|1⟩) = {bob_1/total:.3f}")
success = abs(bob_0 / total - p0_expected) < 0.05
print(f"\nTeleportation {'SUCCESSFUL ✓' if success else 'check failed (try more shots)'}")